# Comparacao de mIOU na primeira iteracao

Este notebook plota o mIOU (conjunto de teste) da **primeira iteracao** para as versoes disponiveis em `bioflore_data` e em `amazon_data`. As listas `bioflore_versions` e `amazon_versions` facilitam adicionar/remover versoes.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt

# Ajuste aqui as versoes a comparar
bioflore_versions = [
    "v02",
    "vDeepLabv3Vanilla",
    "vLaura",
]

amazon_versions = [
    "13_amazon_data",
    "vDeepLabv3Vanilla_resnet50",
    "vLaura_deeplabv3plus_resnet9",
]

iter_name = "iter_001"  # primeira iteracao com metricas salvas
metric_key_suffix = "/avgIOU"  # campo que queremos coletar

# Considerando que o notebook esta em exploration_notebooks/
repo_root = Path.cwd().parents[0]

In [ ]:
def extract_avg_iou(data: dict) -> float | None:
    for key, value in data.items():
        if key.endswith(metric_key_suffix):
            return float(value)
    return None


def load_bioflore_miou(version: str) -> float | None:
    """Media do mIOU entre regioes na primeira iteracao."""
    base = repo_root / "bioflore_data" / version / iter_name
    region_files = sorted(base.glob("region_*/test_metrics.yaml"))
    if not region_files:
        return None

    values = []
    for path in region_files:
        with path.open() as f:
            data = yaml.safe_load(f)
        val = extract_avg_iou(data)
        if val is not None:
            values.append(val)

    return float(pd.Series(values).mean()) if values else None


def load_amazon_miou(version: str) -> float | None:
    """mIOU global do conjunto de teste na primeira iteracao."""
    path = repo_root / "amazon_data" / version / iter_name / "test_metrics.yaml"
    if not path.exists():
        return None
    with path.open() as f:
        data = yaml.safe_load(f)
    return extract_avg_iou(data)


def build_df(versions: list[str], loader) -> pd.DataFrame:
    rows = []
    for v in versions:
        miou = loader(v)
        rows.append({"version": v, "miou": miou})
    return pd.DataFrame(rows)


bioflore_df = build_df(bioflore_versions, load_bioflore_miou)
amazon_df = build_df(amazon_versions, load_amazon_miou)

display(bioflore_df)
display(amazon_df)

In [ ]:
def plot_bar(df: pd.DataFrame, title: str):
    filtered = df.dropna()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(filtered["version"], filtered["miou"], color="#4a90e2")
    ax.set_ylabel("mIOU (teste)")
    ax.set_title(title)
    ax.set_ylim(0, max(filtered["miou"]) * 1.1 if not filtered.empty else 1)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.xticks(rotation=20)
    plt.show()


plot_bar(bioflore_df, "Bioflore: mIOU na primeira iteracao (teste)")
plot_bar(amazon_df, "Amazon: mIOU na primeira iteracao (teste)")